# Baroreflex Sensitivity Testing

**Clinical Application:** Assessment of autonomic function and cardiovascular regulation

**Learning Objectives:**
1. Understand baroreflex physiology and clinical significance
2. Simulate baroreceptor firing dynamics
3. Compute baroreflex sensitivity (BRS)
4. Interpret results in clinical contexts

**Clinical Relevance:**
- Post-MI risk stratification (La Rovere et al. 1998)
- Heart failure prognosis
- Autonomic neuropathy assessment
- Syncope evaluation

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from src.autonomic.baroreflex import Baroreceptor, BaroreflexController, compute_baroreflex_sensitivity
from src.validation.benchmarks import PhysiologicalBenchmarks

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 10)
print("✓ Imports successful")

## Part 1: Baroreceptor Firing Dynamics

### Physiological Background

Baroreceptors in carotid sinus and aortic arch sense arterial pressure changes:
- **High pressure** → Increased firing → ↑ Vagal, ↓ Sympathetic → ↓ HR, ↓ BP
- **Low pressure** → Decreased firing → ↓ Vagal, ↑ Sympathetic → ↑ HR, ↑ BP

In [ ]:
# Create baroreceptor model
baroreceptor = Baroreceptor()

# Test across pressure range
pressures = np.linspace(60, 180, 100)
firing_rates = [baroreceptor.compute_firing_rate(p, 0.001) for p in pressures]

# Plot pressure-firing relationship
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(pressures, firing_rates, 'b-', linewidth=2.5)
ax1.axvline(93, color='g', linestyle='--', label='Normal MAP', alpha=0.7)
ax1.axvline(100, color='r', linestyle='--', label='Sigmoid midpoint', alpha=0.7)
ax1.set_xlabel('Mean Arterial Pressure (mmHg)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Firing Rate (spikes/s)', fontsize=12, fontweight='bold')
ax1.set_title('Baroreceptor Pressure-Firing Relationship\n(Chapleau & Abboud 2001)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Derivative (sensitivity)
dp = np.diff(pressures)
dfr = np.diff(firing_rates)
sensitivity = dfr / dp
ax2.plot(pressures[:-1], sensitivity, 'r-', linewidth=2)
ax2.set_xlabel('Pressure (mmHg)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Sensitivity (spikes/s/mmHg)', fontsize=12, fontweight='bold')
ax2.set_title('Baroreceptor Sensitivity', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Firing at 80 mmHg: {baroreceptor.compute_firing_rate(80, 0.001):.1f} spikes/s")
print(f"Firing at 100 mmHg: {baroreceptor.compute_firing_rate(100, 0.001):.1f} spikes/s")
print(f"Firing at 120 mmHg: {baroreceptor.compute_firing_rate(120, 0.001):.1f} spikes/s")

## Part 2: Baroreflex Control Loop

Simulate complete baroreflex arc: Pressure → Baroreceptor → NTS → Autonomic output

In [ ]:
controller = BaroreflexController()

# Simulate pressure ramp
times = np.arange(0, 20, 0.01)
pressures = 93 + 30 * np.sin(2 * np.pi * times / 10)

vagal_outputs = []
sympathetic_outputs = []
heart_rates = []

for t, p in zip(times, pressures):
    v, s = controller.compute_autonomic_output(p, 0.01, t)
    hr = controller.compute_heart_rate_response(p, baseline_hr=105, dt=0.01, t=t)
    vagal_outputs.append(v)
    sympathetic_outputs.append(s)
    heart_rates.append(hr)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(times, pressures, 'b-', linewidth=2)
axes[0].set_ylabel('Pressure\n(mmHg)', fontsize=11, fontweight='bold')
axes[0].set_title('Baroreflex Response to Pressure Changes', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(times, vagal_outputs, 'g-', linewidth=2, label='Vagal')
axes[1].plot(times, sympathetic_outputs, 'r-', linewidth=2, label='Sympathetic')
axes[1].set_ylabel('Autonomic\nOutput', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

axes[2].plot(times, heart_rates, 'purple', linewidth=2)
axes[2].set_ylabel('Heart Rate\n(bpm)', fontsize=11, fontweight='bold')
axes[2].grid(True, alpha=0.3)

# Phase relationship
axes[3].scatter(pressures, heart_rates, c=times, cmap='viridis', s=10, alpha=0.6)
axes[3].set_xlabel('Pressure (mmHg)', fontsize=11, fontweight='bold')
axes[3].set_ylabel('Heart Rate (bpm)', fontsize=11, fontweight='bold')
axes[3].set_title('Pressure-HR Relationship', fontsize=12, fontweight='bold')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Baroreflex demonstrates reciprocal autonomic control")

## Part 3: Clinical Baroreflex Sensitivity Testing

### Sequence Method (La Rovere et al. 1998)

Identify sequences where systolic BP and RR interval change in same direction

In [ ]:
# Simulate spontaneous BP variations
np.random.seed(42)
n_beats = 100
baseline_sbp = 120
baseline_rr = 850  # ms

sbp_variations = baseline_sbp + np.random.randn(n_beats) * 5
rr_variations = baseline_rr + (sbp_variations - baseline_sbp) * 10 + np.random.randn(n_beats) * 5

# Compute BRS
brs_values = []
for i in range(len(sbp_variations) - 3):
    dp = sbp_variations[i+3] - sbp_variations[i]
    drr = rr_variations[i+3] - rr_variations[i]
    if abs(dp) > 1:
        brs = drr / dp
        if 3 < brs < 30:
            brs_values.append(brs)

mean_brs = np.mean(brs_values) if brs_values else 0

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time series
axes[0,0].plot(sbp_variations, 'b-', linewidth=1.5)
axes[0,0].set_ylabel('SBP (mmHg)', fontsize=11, fontweight='bold')
axes[0,0].set_title('Systolic Blood Pressure', fontsize=12, fontweight='bold')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(rr_variations, 'r-', linewidth=1.5)
axes[0,1].set_ylabel('RR Interval (ms)', fontsize=11, fontweight='bold')
axes[0,1].set_title('RR Intervals', fontsize=12, fontweight='bold')
axes[0,1].grid(True, alpha=0.3)

# Scatter plot
axes[1,0].scatter(sbp_variations, rr_variations, alpha=0.6, s=50)
z = np.polyfit(sbp_variations, rr_variations, 1)
p = np.poly1d(z)
axes[1,0].plot(sbp_variations, p(sbp_variations), 'r--', linewidth=2, label=f'BRS = {z[0]:.1f} ms/mmHg')
axes[1,0].set_xlabel('SBP (mmHg)', fontsize=11, fontweight='bold')
axes[1,0].set_ylabel('RR Interval (ms)', fontsize=11, fontweight='bold')
axes[1,0].set_title('Baroreflex Sensitivity', fontsize=12, fontweight='bold')
axes[1,0].legend(fontsize=11)
axes[1,0].grid(True, alpha=0.3)

# BRS distribution
axes[1,1].hist(brs_values, bins=20, edgecolor='black', alpha=0.7)
axes[1,1].axvline(mean_brs, color='r', linestyle='--', linewidth=2, label=f'Mean = {mean_brs:.1f}')
axes[1,1].axvspan(3, 30, alpha=0.2, color='green', label='Normal range')
axes[1,1].set_xlabel('BRS (ms/mmHg)', fontsize=11, fontweight='bold')
axes[1,1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1,1].set_title('BRS Distribution', fontsize=12, fontweight='bold')
axes[1,1].legend(fontsize=10)
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

benchmarks = PhysiologicalBenchmarks()
print(f"\nBaroreflex Sensitivity: {mean_brs:.1f} ms/mmHg")
print(f"Normal range: {benchmarks.baroreflex.brs_normal.min_value:.1f}-{benchmarks.baroreflex.brs_normal.max_value:.1f} ms/mmHg")
print(f"Interpretation: {'Normal' if 3 < mean_brs < 30 else 'Impaired'}")

## Summary

### Clinical Interpretation

**BRS Values:**
- **>12 ms/mmHg**: Normal baroreflex function
- **6-12 ms/mmHg**: Moderately impaired
- **<6 ms/mmHg**: Severely impaired (high risk)

**Clinical Applications:**
1. Post-MI risk stratification
2. Heart failure prognosis  
3. Autonomic neuropathy detection
4. Drug effect assessment

---
© 2025 Multi-Heart-Model Project | MIT License